# ftfy walkthrough

Hands-on tour of [ftfy](https://github.com/rspeer/python-ftfy), the de facto library for repairing mojibake.

Mojibake (文字化け) is text that looks like garbage because it was decoded with the wrong encoding. The classic Brazilian-Portuguese culprit: text encoded in UTF-8 then mistakenly decoded as Latin-1, which produces sequences like `Ã©` instead of `é`.

`corpus-prep` runs `ftfy.fix_text()` as the first stage of `corpus_prep.normalize`, so every document the pipeline emits is already mojibake-free. This notebook shows the building blocks in isolation.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

## 1. Common Brazilian-Portuguese mojibake gallery

Each row below is a real corruption pattern produced when UTF-8-encoded PT-BR is mis-decoded as Latin-1 (the default for many legacy Windows / older PHP / older CSV exports).

In [ ]:
import ftfy

samples = [
    "Ã©",
    "Ã£o",
    "nÃ£o",
    "sÃ£o",
    "informaÃ§Ã£o",
    "comunicaÃ§Ã£o",
    "prefeitura municipal de Picos â€“ atos oficiais",
    "caf\u00c3\u00a9",
]

for raw in samples:
    fixed = ftfy.fix_text(raw)
    print(f"{raw!r:50}  ->  {fixed!r}")

## 2. Diagnostic mode

`fix_and_explain()` returns a tuple of (fixed text, list of explanations). Useful when you suspect a corpus is corrupted and want to see *what* ftfy detected before applying the fix in bulk.

In [ ]:
for raw in samples[:3]:
    result = ftfy.fix_and_explain(raw)
    fixed = result.text
    explanation = result.explanation
    print(f"{raw!r}")
    print(f"  -> {fixed!r}")
    if explanation:
        print("  applied steps:")
        for step in explanation:
            print(f"    - {step}")
    print()

## 3. ftfy inside the corpus-prep normalize pipeline

`corpus_prep.normalize.normalize()` chains five steps:

1. `ftfy.fix_text` (this notebook's topic)
2. Unicode NFC composition
3. Strip control characters (preserving tab / LF / CR)
4. Collapse runs of spaces and tabs into a single space
5. Collapse 3+ newlines into a single paragraph break

In [ ]:
from corpus_prep.normalize import normalize

messy = (
    "  InformaÃ§Ã£o oficial:    a prefeitura nÃ£o atendeu "
    "\x00 a demanda. \n\n\n\n NÃ£o    haverÃ¡ recurso. "
)
print("raw  :", repr(messy))
print("clean:", repr(normalize(messy)))

## 4. Edge cases ftfy will NOT fix

- **Already-correct text** — ftfy is idempotent; it leaves clean text alone.
- **Encodings outside its repertoire** — ftfy specializes in the Latin-1 / Windows-1252 / UTF-8 confusion. It does *not* try to recover Shift-JIS or GB18030 corruption.
- **Truncated multi-byte sequences** — if a UTF-8 byte was actually lost in transit, no library can reconstruct the character; ftfy leaves a Unicode replacement character («�») where appropriate.

In [ ]:
# Idempotent on clean text:
clean = "informa\u00e7\u00e3o oficial"
assert ftfy.fix_text(clean) == clean
print("clean text untouched:", repr(clean))

## Takeaways

- ftfy reverses the most common encoding accidents you'll see in Brazilian official documents.
- `fix_and_explain` lets you audit a sample before committing to a bulk fix.
- `corpus-prep` runs ftfy automatically as part of `normalize()`, so downstream Documents are already mojibake-free.